In [30]:
# the way I generated the PARM predictions/the way they output files each promoter/ism is output in a ENSG_ENST_HGNC..Promoter directory
# each directory contains:
#
# 1. hits_chrom:start-end.txt.gz
# 2. mutagenesis_chrrom:start-end.txt.gz
#
# 1 contains the hocomoco output from PARM's ISM calling, they consider hits as those with an abs(rho) > 0.75
# for this data I want to convert this to be usable with tangermeme plotting structure 
# we'll return a dictionary of [promoter/enhancer][filtered_hits] - we will reformat the DF to the shape:
#                       example_idx 	start 	end 	attribution 	rho 	enhancer_id 	
#
# matching columns are as follows:
# 'name_motif' -> example_idx
# 'start' -> 'start
# 'end' -> end
# 'att' -> attribution
# 'rho' -> rho (shouldn't be needed for plotting but good for sanity checking)
# 'enhancer_id' -> shouldn't really be necessary again as we won't be calling everything together but may be useful
#
# 2 contains the actual predictions from PARM
# we'll save those as 'raw' tensors in a dictionary with enhancer key, tesor value pairs
# we'll also save those as contribution scores for plotting comparisons
# will also be useful to save in seqlet calling compatible tensor to make comparisons between our methods and theirs   

In [31]:
# update 092625, this is fast, we'll run it all in a notebook instead and just save the outputs for plotting and whatnot #
# we'll generate K562 and HepG2 tensors and go from there in another notebook #
# update 112425, we have a new file naming convention and BED file so we need to reqork things a bit #

In [9]:
# import functions
import glob
import torch
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
import pickle
from collections import Counter
# tangermeme packages
from tangermeme.seqlet import recursive_seqlets
from tangermeme.plot import plot_logo
from tangermeme.annotate import annotate_seqlets
from tangermeme.io import read_meme

In [10]:
# define function for generating an interval : annotation dictionary from a input bed file
# this will be used in both filtering functions to 
def interval_dict (path2bed):
    # open bed file as dataframe
    bed_df = pd.read_csv(path2bed, sep = '\t', header = None)
    # convert the chrom, start, end into key
    bed_key = [f'{id}::{chrom}:{str(start)}-{str(end)}' for chrom, start, end, id in zip(bed_df[0], bed_df[1], bed_df[2], bed_df[3])]
    # get ids for each interval
    gene_vals = list(bed_df[3])
    # make dictionary and return
    return dict(zip(bed_key, gene_vals))


In [11]:
ann_dict = interval_dict('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/raw_data/reformatted_bed_files/gencode.v44.protein.coding.250bp.promoters.autosomes.v2.full.ID.bed')

In [12]:
promoter_bed = pd.read_csv('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/raw_data/reformatted_bed_files/gencode.v44.protein.coding.250bp.promoters.autosomes.v2.full.ID.bed', sep = '\t', header = None)

In [13]:
def promoter_dict (path2bed):
    # open bed file as dataframe
    bed_df = pd.read_csv(path2bed, sep = '\t', header = None)
    # convert the chrom, start, end into key
    bed_vals = [f'{chrom}:{str(start)}-{str(end)}' for chrom, start, end in zip(bed_df[0], bed_df[1], bed_df[2])]
    # get ids for each interval
    gene_keys = [id for id in bed_df[3]]
    # make dictionary and return
    return dict(zip(gene_keys, bed_vals))

In [14]:
promoter_bed_dict = promoter_dict('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/raw_data/reformatted_bed_files/gencode.v44.protein.coding.250bp.promoters.autosomes.v2.full.ID.bed')

In [15]:
# get all k562 predictions
k562_folders = glob.glob('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/PARM/preds_out/all_gencode_v44_promoter_sat_mut_parm_preds_k562/*')
# get all hepg2 predictions
hepg2_folders = glob.glob('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/PARM/preds_out/all_gencode_v44_promoter_sat_mut_parm_preds_hepg2/*')

In [16]:
### let's define the tensor output function ###
### PARM uses REF - ALT for skew calculations, no need to multiply ###
def parm2tangermeme_tensors (path2folders):
    # define dictionaries
    raw_tensor_dict = {}
    seqlet_tensor_dict = {}
    plotLogo_tensor_dict = {}
    oneHot_tensor_dict = {}
    for folder in tqdm(path2folders):
        # get current wd and store as variable for returning to root of folders
        # pull the enhancer/gene name
        element = folder.split('/')[-1].split('::')[0].split('_')[0]
        # get files - these are the same in all outputs and sorted will alphabetize
        # 'hits' ie the TF calls will be index 0
        # 'mutagenesis' ie the actual predictions will be index -1
        parm_files = sorted(os.listdir(folder))
        # get the mutagenesis file for that folder and open as a dataframe
        mut_file = pd.read_csv(f'{folder}/{parm_files[-1]}', sep = '\t')
        feature_cols = ['A', 'C', 'G', 'T']
        # Loop through each possible base
        for base in feature_cols:
            # Find rows where the 'Ref' column matches the current base
            # and set the value in the column with that base's name to 0
            mut_file.loc[mut_file['Ref'] == base, base] = 0
        # generate the 'raw' tensors by converting df to tensor
        raw_tensor = torch.tensor(mut_file.filter(['A', 'C', 'G', 'T']).to_numpy()).T
        # make seqlet tensor
        seqlet_tensor = raw_tensor.sum(dim=0) / 3
        # make oneHot tensor
        oneHot_bool = raw_tensor == 0
        oneHot_tensor = 1 * oneHot_bool
        # make plotLogo tensor
        plotLogo_tensor = seqlet_tensor * oneHot_tensor

        # update dictionaries
        raw_tensor_dict[element] = raw_tensor
        seqlet_tensor_dict[element] = seqlet_tensor
        plotLogo_tensor_dict[element] = plotLogo_tensor
        oneHot_tensor_dict[element] = oneHot_tensor
    return raw_tensor_dict, seqlet_tensor_dict, plotLogo_tensor_dict, oneHot_tensor_dict;

In [17]:
# generate all tangermeme seqlet-necessary tensors for K562
k562_raw_tensors, k562_seqlet_tensors, k562_plotLogo_tensors, k562_oneHot_tensors = parm2tangermeme_tensors(k562_folders)

100%|██████████| 18761/18761 [01:31<00:00, 205.76it/s]


In [18]:
# generate all tangermeme seqlet-necessary tensors for HepG2
hepg2_raw_tensors, hepg2_seqlet_tensors, hepg2_plotLogo_tensors, hepg2_oneHot_tensors = parm2tangermeme_tensors(hepg2_folders)

100%|██████████| 18761/18761 [01:28<00:00, 211.60it/s]


In [19]:
# 'name_motif' -> example_idx
# 'start' -> 'start
# 'end' -> end
# 'att' -> attribution
# 'rho' -> rho (shouldn't be needed for plotting but good for sanity checking)
# 'enhancer_id' -> shouldn't really be necessary again as we won't be calling everything together but may be useful

In [20]:
def map_relative_to_genomic(df, interval_dict):
    """
    Maps relative start/end coordinates in a DataFrame to absolute
    genomic coordinates using an interval dictionary.

    Args:
        df (pd.DataFrame): DataFrame with 'promoter_id', 'start', and 'end' columns.
        interval_dict (dict): Dictionary mapping promoter_id to a genomic
                              interval string (e.g., 'chr:start-end').

    Returns:
        pd.DataFrame: A new DataFrame with added 'chrom', 'genomic_start',
                      and 'genomic_end' columns.
    """
    # Create a new DataFrame to avoid modifying the original
    new_df = df.copy()

    # Extract promoter coordinates from the dictionary
    # This uses the .map() function for an efficient lookup
    interval_series = new_df['promoter_id'].map(interval_dict)

    # Split the interval string 'chr:start-end' into three parts
    coords_split = interval_series.str.split('[:-]', expand=True)

    # Assign new columns for chromosome and the promoter's start
    new_df['chrom'] = coords_split[0]
    promoter_start = pd.to_numeric(coords_split[1])

    # Calculate the absolute genomic coordinates
    new_df['genomic_start'] = promoter_start + new_df['start']
    new_df['genomic_end'] = promoter_start + new_df['end']

    return new_df

In [21]:
# define function for storing significant PARM calls as dictionaries and save as pickles for plotting/comparisons etc.
def filterPARM_tf_calls (path2folders,
                         annotation_dict,
                         pro_dict):
    # make a dictionary for storing calls as a vector
    # index 0: raw parm output
    # index 1: tangermeme_reformatted parm output
    # index 2: all of the above with genomic intervals
    # index 3: only significant parm hits, abs(rho) > 0.75
    tf_call_dict = {}
    for folder in tqdm(path2folders):
        # get current wd and store as variable for returning to root of folders
        # get the interval from the folder name
        interval = folder.split('/')[-1]
        # pull the enhancer/gene name
        interval_name = annotation_dict.get(interval)
        # get files - these are the same in all outputs and sorted will alphabetize
        # 'hits' ie the TF calls will be index 0
        # 'mutagenesis' ie the actual predictions will be index -1
        parm_files = sorted(os.listdir(folder))
        # get the mutagenesis file for that folder and open as a dataframe
        tf_file = pd.read_csv(f'{folder}/{parm_files[0]}', sep = '\t')
        # make a 'new' df compatible with tangermeme anontations
        reformatted_df = pd.DataFrame({'example_idx' : tf_file['name_motif'],
                                       'start' : tf_file['start'],
                                       'end' : tf_file['end'],
                                       'attribution' : tf_file['att'],
                                       'rho' : tf_file['rho'],
                                       'promoter_id' : [interval_name for i in range(len(tf_file))]})
        # return a df with genomic intervals
        genomic_tfs = map_relative_to_genomic(reformatted_df, pro_dict)
        # make a bool for filtering significant hits
        sig_hit_bool = abs(genomic_tfs['rho']) > 0.75
        sig_tfs = genomic_tfs.loc[sig_hit_bool]
        # update dictionary
        tf_call_dict[interval_name.split('_')[0]] = (tf_file, reformatted_df, genomic_tfs, sig_tfs)
    return tf_call_dict

In [22]:
# get TF calls for K562
k562_parmTF_calls = filterPARM_tf_calls(k562_folders,
                                        ann_dict,
                                        promoter_bed_dict)

100%|██████████| 18761/18761 [03:12<00:00, 97.60it/s] 


In [23]:
# get TF calls for HepG2
hepg2_parmTF_calls = filterPARM_tf_calls(hepg2_folders,
                                         ann_dict,
                                         promoter_bed_dict)

100%|██████████| 18761/18761 [03:09<00:00, 98.99it/s] 


In [24]:
# store the promoter keys as list for a fixed order of iterating through things
promoter_list = list(ann_dict.values())

In [25]:
# define a function for returning a dataframe of predictions for generating scatterplots across models
def PARM2DF (path2folders,
             annotation_dict,
             cell_type):
    
    # make list for storing tiny DFs for concatenating
    dfs2cat_all = []
    for folder in tqdm(path2folders):
        element2cat = []
        # get the interval from the folder name
        interval = folder.split('/')[-1].split('::')[-1]
        # break the interval name into chromosome and pos
        chrom = interval.split(':')[0]
        pos = int(interval.split('-')[0].split(':')[-1]) + 1 # we add 1 to the start interval of BED files for converting to VCF
        # get mutation files
        parm_files = sorted(os.listdir(folder))
        # get the mutagenesis file for that folder and open as a dataframe
        mut_file = pd.read_csv(f'{folder}/{parm_files[-1]}', sep = '\t')
        feature_cols = ['A', 'C', 'G', 'T']
        # Loop through each possible base
        for base in feature_cols:
            # Find rows where the 'Ref' column matches the current base
            # and set the value in the column with that base's name to 0
            mut_file.loc[mut_file['Ref'] == base, base] = 0
        # add chromosome and position information
        mut_file.loc[:,'chrom'] = [chrom for i in range(len(mut_file))]
        pos2add = []
        for position in range(len(mut_file)):
            pos2add.append(pos)
            pos+=1
        mut_file.loc[:,'pos'] = pos2add
        # Melt the dataframe
        mut_file_long = mut_file.melt(
            id_vars=['chrom', 'pos', 'Ref'],
            value_vars=['A', 'C', 'G', 'T'],
            var_name='alt',      # New column for the base names
            value_name='skew'    # New column for the scores
        )
        # Rename 'Ref' to 'ref' to match your requested format
        mut_file_long.rename(columns={'Ref': 'ref'}, inplace=True)
        # Sort the values for better readability
        mut_file_long.sort_values(by=['pos', 'alt'], inplace=True)
        # Create the final DataFrame by filtering out matching ref/alt pairs
        final_df = mut_file_long[mut_file_long['ref'] != mut_file_long['alt']].copy()
        # add element name
        final_df.loc[:,'gene_id'] = [folder.split('/')[-1].split('::')[0].split('_')[0] for i in range(len(final_df))]
        final_df.loc[:, 'merge_id'] = [(':').join([chrom, str(pos), ref, alt]) for chrom, pos, ref, alt in zip(final_df['chrom'], final_df['pos'], final_df['ref'], final_df['alt'])]
        final_df.reset_index(drop=True, inplace=True)
        dfs2cat_all.append(final_df)
    return pd.concat(dfs2cat_all)

In [26]:
k562_parm_preds_reformat = PARM2DF(k562_folders,
                                   ann_dict,
                                   'k562')

  0%|          | 78/18761 [00:00<02:29, 124.92it/s]

100%|██████████| 18761/18761 [02:34<00:00, 121.14it/s]


In [27]:
hepg2_parm_preds_reformat = PARM2DF(hepg2_folders,
                                   ann_dict,
                                   'hepg2')

100%|██████████| 18761/18761 [02:40<00:00, 117.04it/s]


In [28]:
# all reformatted preictions have length 750... except... IPMK which contains an N, see below
# ipmk = hepg2_parm_preds_reformat[hepg2_parm_preds_reformat['id'] == 'IPMK']
# ipmk[ipmk['pos'] == 58267943]
# maybe drop that one from downstream analyses, but everything else looks good!

In [17]:
# save k562 stuff to disk
# raw tensors
torch.save(k562_raw_tensors, '../PARM/processed_data/k562_parm_raw_tensors.pt')
# seqlet tensors
torch.save(k562_seqlet_tensors, '../PARM/processed_data/k562_parm_seqlet_tensors.pt')
# plotLogo tensors
torch.save(k562_plotLogo_tensors, '../PARM/processed_data/k562_parm_plotLogo_tensors.pt')
# oneHot tensors
torch.save(k562_oneHot_tensors, '../PARM/processed_data/k562_parm_oneHot_tensors.pt')
# parm TF calls
torch.save(k562_parmTF_calls, '../PARM/processed_data/k562_parm_TFs.pt')
# predictions
#k562_parm_preds_reformat.to_csv('../PARM/processed_data/k562_parm_preds.tsv.gz', sep = '\t', index = False, compression='gzip')

In [19]:
# save hepg2 stuff to disk
# raw tensors
torch.save(hepg2_raw_tensors, '../PARM/processed_data/hepg2_parm_raw_tensors.pt')
# seqlet tensors
torch.save(hepg2_seqlet_tensors, '../PARM/processed_data/hepg2_parm_seqlet_tensors.pt')
# plotLogo tensors
torch.save(hepg2_plotLogo_tensors, '../PARM/processed_data/hepg2_parm_plotLogo_tensors.pt')
# oneHot tensors
torch.save(hepg2_oneHot_tensors, '../PARM/processed_data/hepg2_parm_oneHot_tensors.pt')
# parm TF calls
torch.save(hepg2_parmTF_calls, '../PARM/processed_data/hepg2_parm_TFs.pt')
# predictions
#hepg2_parm_preds_reformat.to_csv('../PARM/processed_data/hepg2_parm_preds.tsv.gz', sep = '\t', index = False, compression='gzip')

In [29]:
# store the dictionary keys (promoter names) as a list for annotating seqlets, oneHots, etc.
promoter_list = list(k562_raw_tensors.keys())

In [30]:
# define a function for stacking each seqlet dictionary for each cell type
def stack_tensors (seqlet_dict, plotLogo_dict, oneHot_dict, promoters):
    # stack seqlets
    seqlet_stack = torch.stack([seqlet_dict[promoter] for promoter in promoters])
    # stack plotLogos
    plotLogo_stack = torch.stack([plotLogo_dict[promoter] for promoter in promoters])
    # stack oneHots
    oneHot_stack = torch.stack([oneHot_dict[promoter] for promoter in promoters])
    return seqlet_stack, plotLogo_stack, oneHot_stack;

In [31]:
# stack all tensors for calling seqlets, annotating them, etc.
# k562
k_stackedSeqlets, k_stackedPlotLogos, k_stackedOneHots = stack_tensors(k562_seqlet_tensors, k562_plotLogo_tensors, k562_oneHot_tensors, promoter_list)
# hepg2
h_stackedSeqlets, h_stackedPlotLogos, h_stackedOneHots = stack_tensors(hepg2_seqlet_tensors, hepg2_plotLogo_tensors, hepg2_oneHot_tensors, promoter_list)

In [32]:
# call seqlets for each cell type
# k562
k_seqlet_calls = recursive_seqlets(k_stackedSeqlets, threshold=0.01)
# hepg2
h_seqlet_calls = recursive_seqlets(h_stackedSeqlets, threshold=0.01)

In [33]:
# define a function for annotating seqlets
def ann_seqlets (path2pwms, motif_database, stacked_oneHots, seqlet_calls, promoters):
    # store motif info as a variable
    motif_path = f'{path2pwms}/{motif_database}'
    # open pwms
    motifs = read_meme(motif_path)
    motif_names = list(motifs.keys())
    # run annotate seqlets
    motif_idxs, motif_pvals = annotate_seqlets(stacked_oneHots, seqlet_calls, motif_path)
    # make a dictionary of indices and gene names for annotating promoters
    promoter_idx_dict = dict(zip([promoters.index(i) for i in promoters], promoters))
    # make copies of the seqlet DFs for annotating with promoter names and tf names
    # tf for plotLogos
    tf_ann_seqlets = seqlet_calls.copy()
    tf_ann_seqlets.loc[:,'example_idx'] = [motif_names[idx] for idx in motif_idxs] # use this one with plotLogos
    # tf + promoter id
    full_ann_seqlets = seqlet_calls.copy()
    # add enhancer ids
    full_ann_seqlets.loc[:, 'promoter_id'] = [promoter_idx_dict.get(i) for i in full_ann_seqlets['example_idx']]
    # add matching motif
    full_ann_seqlets.loc[:,'example_idx'] = [motif_names[idx] for idx in motif_idxs]
    # add pval of match
    full_ann_seqlets.loc[:,'pval'] = [float(i[0]) for i in motif_pvals]
    return tf_ann_seqlets, full_ann_seqlets;

In [34]:
# annotate seqlets for each cell type
# k562
k_tf_ann_seqlets, k_full_ann_seqlets = ann_seqlets('/projects/tewhey-lab/buttsj/meme_suite_data/motif_databases/HOCOMOCO/',
                                                        'HOCOMOCOv11_core_HUMAN_mono_meme_format.meme',
                                                        k_stackedOneHots,
                                                        k_seqlet_calls,
                                                        promoter_list)
# hepg2
h_tf_ann_seqlets, h_full_ann_seqlets = ann_seqlets('/projects/tewhey-lab/buttsj/meme_suite_data/motif_databases/HOCOMOCO/',
                                                        'HOCOMOCOv11_core_HUMAN_mono_meme_format.meme',
                                                        h_stackedOneHots,
                                                        h_seqlet_calls,
                                                        promoter_list)

/tmp/ipykernel_1857773/3739010979.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['ELK1_HUMAN.H11MO.0.B', 'TYY1_HUMAN.H11MO.0.A', 'GABPA_HUMAN.H11MO.0.A', 'THA11_HUMAN.H11MO.0.B', 'ATF4_HUMAN.H11MO.0.A', 'TYY1_HUMAN.H11MO.0.A', 'THA11_HUMAN.H11MO.0.B', 'KAISO_HUMAN.H11MO.0.A', 'ELK1_HUMAN.H11MO.0.B', 'NRF1_HUMAN.H11MO.0.A', 'FOXI1_HUMAN.H11MO.0.B', 'ELK1_HUMAN.H11MO.0.B', 'SNAI1_HUMAN.H11MO.0.C', 'THA11_HUMAN.H11MO.0.B', 'THA11_HUMAN.H11MO.0.B', 'ELK1_HUMAN.H11MO.0.B', 'SNAI1_HUMAN.H11MO.0.C', 'SNAI1_HUMAN.H11MO.0.C', 'THA11_HUMAN.H11MO.0.B', 'NRF1_HUMAN.H11MO.0.A', 'ELK1_HUMAN.H11MO.0.B', 'ELK1_HUMAN.H11MO.0.B', 'KAISO_HUMAN.H11MO.0.A', 'SP1_HUMAN.H11MO.0.A', 'ELK1_HUMAN.H11MO.0.B', 'RORG_HUMAN.H11MO.0.C', 'THA11_HUMAN.H11MO.0.B', 'NRF1_HUMAN.H11MO.0.A', 'SP1_HUMAN.H11MO.0.A', 'NRF1_HUMAN.H11MO.0.A', 'ELK1_HUMAN.H11MO.0.B', 'ELK1_HUMAN.H11MO.0.B', 'ELF1_HUMAN.H11MO.0.A', 'ZN143_HUMAN.H11MO.0.A', 'ELK1_HUMAN.

In [ ]:
# save annotated seqlets and things needed for plotting to disk
# k562
k_seqlet_calls.to_csv('../PARM/processed_data/parm_k562_tangermeme_seqlet_calls.tsv', sep = '\t', index = False)
k_full_ann_seqlets.to_csv('../PARM/processed_data/parm_k562_tangermeme_full_ann_seqlets.tsv', sep = '\t', index = False)
with open('../PARM/processed_data/parm_k562_tangermeme_plotLogo_dict.pkl', 'wb') as f: # Open in write binary mode ('wb')
    pickle.dump(k562_plotLogo_tensors, f)
# hepg2
h_seqlet_calls.to_csv('../PARM/processed_data/parm_hepg2_tangermeme_seqlet_calls.tsv', sep = '\t', index = False)
h_full_ann_seqlets.to_csv('../PARM/processed_data/parm_hepg2_tangermeme_full_ann_seqlets.tsv', sep = '\t', index = False)
with open('../PARM/processed_data/parm_hepg2_tangermeme_plotLogo_dict.pkl', 'wb') as f: # Open in write binary mode ('wb')
    pickle.dump(hepg2_plotLogo_tensors, f)

In [38]:
# define the optimized function for converting seqlets to a bed file
def seqlets2bed_optimized(full_preds,
                          full_ann_seqlets):
    """
    Converts seqlet predictions to a BED DataFrame using vectorized operations.
    """
    # 1. Get min position for each enhancer. This is already efficient.
    # The result is a Series where the index is 'id' and values are the min 'pos'.
    promoter_min_pos = full_preds.groupby('gene_id')['pos'].min()
    promoter_chrom = full_preds.groupby('gene_id')['chrom'].min()
    #promoter_attrib = 
    # Get the seqlets DataFrame from your dictionary
    seqlets = full_ann_seqlets.copy() # Use .copy() to avoid SettingWithCopyWarning

    # 2. Map the enhancer start positions to each seqlet.
    # This single vectorized operation replaces your entire outer 'for enh...' loop.
    seqlets['promoter_start'] = seqlets['promoter_id'].map(promoter_min_pos)
    seqlets['chrom'] = seqlets['promoter_id'].map(promoter_chrom)
    # Drop any seqlets whose enhancer_id was not found in 'full_preds'
    seqlets.dropna(subset=['promoter_start'], inplace=True)
    
    # Ensure the new column is an integer type for calculations
    seqlets['promoter_start'] = seqlets['promoter_start'].astype(int)

    # 3. Calculate BED coordinates and IDs vectorially.
    # These three lines replace your inner loop and all list appends.
    bed_start = seqlets['promoter_start'] + seqlets['start'] - 1 # Convert to 0-based
    bed_end = seqlets['promoter_start'] + seqlets['end']
    bed_id = seqlets['example_idx'].astype(str) + '_' + seqlets['promoter_id']
    # append attribution to id
    bed_id = [f'{i}_{str(j)}' for i, j in zip(bed_id, seqlets['attribution'].tolist())]
    chromosome = seqlets['chrom']
    # 4. Assemble the final DataFrame in one step.
    bed2return = pd.DataFrame({
        'chrom': chromosome,
        'start': bed_start,
        'end': bed_end,
        'id': bed_id
    }).sort_values(by=['chrom', 'start'])
    
    return bed2return

In [39]:
# generate BED files for each cell type
# k562
k_seqlet_bed = seqlets2bed_optimized(k562_parm_preds_reformat, k_full_ann_seqlets)
# hepg2
h_seqlet_bed = seqlets2bed_optimized(hepg2_parm_preds_reformat, h_full_ann_seqlets)

In [43]:
k_seqlet_bed['id'].tolist()[0]

'SP1_HUMAN.H11MO.0.A_ENSG00000188976.11_-3.420764602127747'

In [42]:
# save BED files to disk
# k562
k_seqlet_bed.to_csv('../processed_data/bed_files/k562_parm_annotated_seqlets_01_112225.bed', sep = '\t', index = False, header = None)
# hepg2
h_seqlet_bed.to_csv('../processed_data/bed_files/hepg2_parm_annotated_seqlets_01_112225.bed', sep = '\t', index = False, header = None)

In [54]:
# reformat (again) reformatted predictions to match VCF format for annotating predictions with seqlets with vcfannotatefrombed
k562_parm_vcf = pd.DataFrame({'#CHROM' : k562_parm_preds_reformat['chrom'],
                              'POS' : k562_parm_preds_reformat['pos'],
                              'ID' : k562_parm_preds_reformat['gene_id'],
                              'REF' : k562_parm_preds_reformat['ref'],
                              'ALT' : k562_parm_preds_reformat['alt'],
                              'QUAL' : ['.' for i in range(len(k562_parm_preds_reformat))],
                              'FILTER' : ['.' for i in range(len(k562_parm_preds_reformat))],
                              'INFO' : k562_parm_preds_reformat['skew']})
hepg2_parm_vcf = pd.DataFrame({'#CHROM' : hepg2_parm_preds_reformat['chrom'],
                               'POS' : hepg2_parm_preds_reformat['pos'],
                               'ID' : hepg2_parm_preds_reformat['gene_id'],
                               'REF' : hepg2_parm_preds_reformat['ref'],
                               'ALT' : hepg2_parm_preds_reformat['alt'],
                               'QUAL' : ['.' for i in range(len(hepg2_parm_preds_reformat))],
                               'FILTER' : ['.' for i in range(len(hepg2_parm_preds_reformat))],
                               'INFO' : hepg2_parm_preds_reformat['skew']})
# save to disk
k562_parm_vcf.to_csv('../processed_data/all_k562_250bp_parm_preds_fullIDs.vcf', sep = '\t', index = False)
hepg2_parm_vcf.to_csv('../processed_data/all_hepg2_250bp_parm_preds_fullIDs.vcf', sep = '\t', index = False)

In [55]:
# concatenate all significant TF calls for each cell type for merging bed file and annotating predictions in seqlets
k562_tfs2cat = []
hepg2_tfs2cat = []
# get k562 tf calls
for promoter in k562_parmTF_calls.keys():
    sig_hits = k562_parmTF_calls.get(promoter)[-1]
    k562_tfs2cat.append(sig_hits)
# get hepg2 tf calls
for promoter in hepg2_parmTF_calls.keys():
    sig_hits = hepg2_parmTF_calls.get(promoter)[-1]
    hepg2_tfs2cat.append(sig_hits)
# concatenate calls
all_k562_hits = pd.concat(k562_tfs2cat)
all_hepg2_hits = pd.concat(hepg2_tfs2cat)
# reformat bed file to only have genomic coordinates for BED file
all_k562_bed = pd.DataFrame({0 : all_k562_hits['chrom'],
                             1 : all_k562_hits['genomic_start'],
                             2 : all_k562_hits['genomic_end'],
                             3 : [f'{tf}:{rho}:{idee}' for tf, rho, idee in zip(all_k562_hits['example_idx'], all_k562_hits['rho'], all_k562_hits['promoter_id'])]})
all_hepg2_bed = pd.DataFrame({0 : all_hepg2_hits['chrom'],
                              1 : all_hepg2_hits['genomic_start'],
                              2 : all_hepg2_hits['genomic_end'],
                              3 : [f'{tf}:{rho}:{idee}' for tf, rho, idee in zip(all_hepg2_hits['example_idx'], all_hepg2_hits['rho'], all_hepg2_hits['promoter_id'])]})
# save BEDs to disk - need to be sorted on CLI !
all_k562_bed.to_csv('../processed_data/bed_files/parm_k562_sig_tf_hits_all.bed', sep = '\t', index = False, header = None)
all_hepg2_bed.to_csv('../processed_data/bed_files/parm_hepg2_sig_tf_hits_all.bed', sep = '\t', index = False, header = None)

In [56]:
all_k562_bed.head()

,0,1,2,3
9,chr1,27959820,27959829,ZBT7A_HUMAN.H11MO.0.A-:-0.7590842974821838:ENS...
14,chr1,27959921,27959930,ZIC1_HUMAN.H11MO.0.B:0.7611663504177488:ENSG00...
23,chr1,27959921,27959931,KLF4_HUMAN.H11MO.0.A:0.7597903868504254:ENSG00...
24,chr1,27959937,27959947,KLF4_HUMAN.H11MO.0.A-:0.7822959703067957:ENSG0...
28,chr1,27959803,27959814,ZN449_HUMAN.H11MO.0.C:0.7761626578006116:ENSG0...


In [44]:
# open merged BEDs from 'sort_and_merge_parm_tf_calls.sh' and save as 'annotation version' ie 1s in INFO column for annotating predictions
k562_merged_parm_tf_bed = pd.read_csv('../processed_data/bed_files/PARM_K562_sig_TFs_bedOps_merged_noMin.bed', sep = '\t', header = None)
hepg2_merged_parm_tf_bed = pd.read_csv('../processed_data/bed_files/PARM_HepG2_sig_TFs_bedOps_merged_noMin.bed', sep = '\t', header = None)
# refactor info column for annotation
k562_merged_parm_tf_bed[3] = [1 for i in range(len(k562_merged_parm_tf_bed))]
hepg2_merged_parm_tf_bed[3] = [1 for i in range(len(hepg2_merged_parm_tf_bed))]
# save to disk
k562_merged_parm_tf_bed.to_csv('../processed_data/bed_files/PARM_K562_TFs_4_annotation.bed', sep = '\t', index = False, header = None)
hepg2_merged_parm_tf_bed.to_csv('../processed_data/bed_files/PARM_HepG2_TFs_4_annotation.bed', sep = '\t', index = False, header = None)